# Week 10 · Day 2 — Inside the Transformer (Guided Notebook — Student)
## SEE attention · meet the decoder

**Time:** ~20 minutes · **You'll run four demos, one per big idea from the lecture:**

1. **The meaning-space** — measure it yourself; watch *cat / kitten / dog* cluster and *airplane* sit alone.
2. **SEE attention** — pull the real attention weights out of DistilBERT and watch **"it" reach back to "mat"** — then flip between heads.
3. **Meet the decoder** — run **GPT-2**'s autoregressive loop by hand, one word at a time, and see the 50,000-way probability list it picks from.
4. **Read-then-write** — hand **T5** a paragraph and watch it summarize.

> Everything here is the **same attention mechanism** you learned yesterday, rewired three ways. Nothing new to build — you're here to *see* it.

**Heads-up:** the first run downloads three small models (DistilBERT, GPT-2, T5-small — a few hundred MB total). If a cell is slow the first time, that's the download; it's cached after.

In [ ]:
# Setup — import once. (transformers + torch were installed for Week 10.)
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, pipeline

torch.manual_seed(42)  # reproducible generation
print('torch', torch.__version__, '| ready')

---
## Demo 1 · The meaning-space is real

In the primer you were told: *each word ID looks up a vector, and similar words get similar vectors.*
Let's **measure** it. We'll grab DistilBERT's embedding table, look up a few words, and compute how
close each pair is with **cosine similarity** (1.0 = identical direction, 0 = unrelated).

In [ ]:
# Load DistilBERT's tokenizer and its learned embedding table (no training — just the lookup).
tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')
emb_model = AutoModel.from_pretrained('distilbert-base-uncased')
embedding_table = emb_model.get_input_embeddings().weight  # [vocab_size, 768]
print('embedding table shape:', tuple(embedding_table.shape))

In [ ]:
def word_vector(word):
    """Average the (sub)word embedding rows for a word -> one 768-dim meaning vector."""
    ids = tok(word, add_special_tokens=False)['input_ids']
    return embedding_table[ids].mean(dim=0).detach()

def cosine(a, b):
    # TODO: return the cosine similarity of vectors a and b (hint: torch.nn.functional.cosine_similarity(..., dim=0)).item()
    return torch.nn.functional.cosine_similarity(a, b, dim=0).item()

words = ['cat', 'kitten', 'dog', 'run', 'ran', 'airplane']
vecs = {w: word_vector(w) for w in words}

# Build the similarity matrix
sim = np.array([[cosine(vecs[a], vecs[b]) for b in words] for a in words])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim, cmap='viridis', vmin=0, vmax=1)
ax.set_xticks(range(len(words))); ax.set_xticklabels(words, rotation=45, ha='right')
ax.set_yticks(range(len(words))); ax.set_yticklabels(words)
for i in range(len(words)):
    for j in range(len(words)):
        ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center',
                color='white' if sim[i,j] < 0.6 else 'black', fontsize=9)
ax.set_title('Cosine similarity in meaning-space')
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

**Read the grid.** The warm block in the top-left (cat / kitten / dog) is the animals clustering.
`run` / `ran` form their own warm pair. `airplane` is cold against everything — the loner.

> **Nobody programmed this.** DistilBERT discovered the geometry of meaning just by reading text —
> exactly like a Week 8 CNN discovered edge filters nobody coded.

**Your turn:** add two words of your own to the `words` list (try `'puppy'` and `'rocket'`) and re-run.
Does `puppy` join the animals? Does `rocket` sit near `airplane`?

---
## Demo 2 · SEE attention — watch "it" find "mat"

Yesterday attention was a recipe on paper. Now we pull the **actual attention weights** out of the
model and plot them. Bright cell = 'this word paid a lot of attention to that word.'

We load the model with `attn_implementation='eager'` — that's the switch that makes it **hand back the
attention weights** so we can look at them.

In [ ]:
# Reload DistilBERT so it returns attention weights.
attn_model = AutoModel.from_pretrained('distilbert-base-uncased', attn_implementation='eager')
attn_model.eval()

sentence = 'The cat sat on the mat because it was soft.'
inputs = tok(sentence, return_tensors='pt')
tokens = tok.convert_ids_to_tokens(inputs['input_ids'][0])

with torch.no_grad():
    out = attn_model(**inputs, output_attentions=True)

# out.attentions: tuple(num_layers) of [batch, num_heads, seq, seq]
print('layers:', len(out.attentions), '| heads per layer:', out.attentions[0].shape[1])
print('tokens:', tokens)

In [ ]:
def show_attention(layer, head):
    A = out.attentions[layer][0, head].numpy()  # [seq, seq]
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(A, cmap='magma')
    ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=90)
    ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
    ax.set_xlabel('attending TO (key)'); ax.set_ylabel('attending FROM (query)')
    ax.set_title(f'DistilBERT attention — layer {layer}, head {head}')
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout(); plt.show()

# A layer/head where coreference tends to show up nicely. Try others!
# TODO: pick a layer (0-5) and head (0-11) to view. Try to find one where 'it' points at 'mat'.
show_attention(layer=4, head=7)

**Find the row for `it`.** Slide along it and look for the brightest square — for many heads it lands on
`mat` (or `cat`). The model learned to make a pronoun *look at what it refers to*, with nobody telling it
what a pronoun is.

Notice too: the bright **diagonal** (every word attends to itself) and that **`[CLS]`** (front) often
glows — the 'resting spot' from the lecture.

**Your turn — see the committee.** Change `head` (0–11) and `layer` (0–5) and re-run. Some heads are pure
**local** stripes (each word → its neighbors); others make **long reaches** across the sentence. Same model,
different specialists.

In [ ]:
# Compare two heads side by side to feel the 'committee of specialists'.
for L, H in [(1, 3), (4, 7)]:
    show_attention(L, H)

---
## Demo 3 · Meet the decoder — GPT-2 writes one word at a time

DistilBERT *reads* (encoder). **GPT-2** *writes* (decoder). Same attention machinery, but with the
**causal mask** — each word may only look left. Generation is a **loop**: predict one token, append it,
feed the whole thing back, predict again. Let's run that loop **by hand** and watch it happen.

In [ ]:
gpt2_tok = AutoTokenizer.from_pretrained('gpt2')
gpt2 = AutoModelForCausalLM.from_pretrained('gpt2')
gpt2.eval()
print('GPT-2 loaded —', gpt2.num_parameters()//1_000_000, 'M parameters (tiny by modern standards)')

In [ ]:
# The autoregressive loop, step by step (greedy: always take the highest-probability token).
prompt = 'The cat sat on the'
ids = gpt2_tok(prompt, return_tensors='pt')['input_ids']

print(f'start: {prompt!r}')
for step in range(8):
    with torch.no_grad():
        logits = gpt2(ids).logits[:, -1, :]      # scores for the NEXT token
    probs = torch.softmax(logits, dim=-1)         # same softmax you've used since Week 6
    next_id = torch.argmax(probs, dim=-1)          # TODO: greedy = the token with the HIGHEST probability
    ids = torch.cat([ids, next_id[None, :]], dim=1)  # append + feed back
    print(f'  step {step+1}: +{gpt2_tok.decode(next_id)!r:12}  ->  {gpt2_tok.decode(ids[0])!r}')

That's **autoregression**: predict → append → feed back → repeat. When ChatGPT types word by word, this
is *literally* what it's doing — not a loading animation.

Now let's open the box on one step: the model doesn't output a *word*, it outputs a **probability over the
whole ~50,000-token vocabulary**. Here are the top 10 candidates for the word after the prompt:

In [ ]:
ids = gpt2_tok(prompt, return_tensors='pt')['input_ids']
with torch.no_grad():
    probs = torch.softmax(gpt2(ids).logits[:, -1, :], dim=-1)[0]
top = torch.topk(probs, 10)
print(f'Top 10 next-word candidates after {prompt!r}:')
for p, i in zip(top.values, top.indices):
    bar = '#' * int(p.item() * 60)
    print(f'  {gpt2_tok.decode(i)!r:14} {p.item()*100:5.1f}%  {bar}')
print('\n...and ~49,990 more tokens, each with a tiny probability.')

> **The knob for Week 11:** greedy (always the top bar) is safe but repetitive. Rolling weighted dice over
> this list — turning **temperature** up or down — is how you get creativity vs. focus. We tune that next week.

**Your turn:** try a different `prompt` (e.g. `'In the year 2050,'`) and re-run the loop. GPT-2 is small, so it
often drifts or repeats — that messiness is *exactly* what scaling the same decoder up fixes.

In [ ]:
# The .generate() method does the loop for you (with nicer sampling). Compare greedy vs. sampled.
ids = gpt2_tok('In the year 2050,', return_tensors='pt')['input_ids']
greedy = gpt2.generate(ids, max_new_tokens=30, do_sample=False, pad_token_id=gpt2_tok.eos_token_id)
sampled = gpt2.generate(ids, max_new_tokens=30, do_sample=True, temperature=0.9, pad_token_id=gpt2_tok.eos_token_id)
print('GREEDY :', gpt2_tok.decode(greedy[0], skip_special_tokens=True))
print()
print('SAMPLED:', gpt2_tok.decode(sampled[0], skip_special_tokens=True))

---
## Demo 4 · Read-then-write — T5 summarizes

The third family: **encoder-decoder**. It **reads** the whole input (encoder), then **writes** a new
sequence (decoder). Summarization is the perfect example — you can't summarize until you've read it all.

In [ ]:
# Encoder-decoder: the encoder READS the whole article, the decoder WRITES a summary.
# We use the low-level model + generate() (works across transformers versions).
# T5 is told the task with a text prefix -- literally the word 'summarize:' in front.
from transformers import AutoModelForSeq2SeqLM

t5_tok = AutoTokenizer.from_pretrained('t5-small')
t5 = AutoModelForSeq2SeqLM.from_pretrained('t5-small')

article = (
    'The transformer architecture, introduced in 2017, replaced recurrent networks for most language '
    'tasks. Its key idea is self-attention, which lets every word directly consider every other word in '
    'the sequence. This made training far more parallel and allowed models to capture long-range '
    'dependencies that older recurrent models struggled with. Scaling transformers up led to the large '
    'language models that power modern chat assistants.'
)
# TODO: T5 needs a task prefix. Put the word 'summarize: ' in front of the article.
input_ids = t5_tok('summarize: ' + article, return_tensors='pt').input_ids
summary_ids = t5.generate(input_ids, max_new_tokens=40, num_beams=2)   # decoder writes, left-to-right
print('SUMMARY:', t5_tok.decode(summary_ids[0], skip_special_tokens=True))

# (On many setups the one-liner also works: pipeline('summarization', model='t5-small').)

---
## Wrap-up · the three families, in one table

| Family | Sees | Job | You met it | Example |
|---|---|---|---|---|
| **Encoder** | every word, both directions | **read / understand** | Demo 2 (yesterday: classify news) | DistilBERT (BERT) |
| **Decoder** | only words to its **left** | **write / generate** | Demo 3 | GPT-2 (GPT) |
| **Encoder-decoder** | read all, then write | **read then write** | Demo 4 | T5 |

**One mechanism, three wirings.** And the middle row — the decoder — scaled up 100–1000×, is an **LLM**.
That's next week.

### Reflection (answer in a sentence each)
1. Why can't a single fixed embedding be right for the word *bank* — and what fixes it?
2. In Demo 2, which token did `it` attend to most? What does that tell you the head learned?
3. In Demo 3, why must the decoder be forbidden from looking to its right during training?